# 05 — Qdrant Vector Indexing

This notebook implements the fifth stage of the **CLARA** ingestion pipeline.

It loads the semantic vectors generated by `04_embedding_generation.ipynb`, creates the corresponding vector collection in **Qdrant**, and uploads each embedding together with its legal metadata.

After this stage, the legal corpus is no longer only a collection of processed files: it becomes a searchable vector database that can be queried using natural-language embeddings.

The default CLARA collection is:

```text
clara_bge_m3
```

## 1. Imports

This block loads the libraries required for filesystem access, JSON processing, HTTP communication, batching, timing, and inspection of the indexing results.

Qdrant is accessed through its HTTP API, which keeps the indexing procedure explicit and independent from a specific Python client implementation.

In [1]:
from pathlib import Path
import json
import uuid

import pandas as pd
import requests


## 2. Project Paths and Qdrant Configuration

The notebook defines the location of the generated embedding files and the configuration required to communicate with Qdrant.

The vector collection name and Qdrant endpoint must remain consistent with the runtime configuration used by the CLARA retrieval application.

In [2]:
PROJECT_ROOT = Path("..").resolve()

EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "clara_bge_m3"

BATCH_SIZE = 64

print(f"Embeddings: {EMBEDDINGS_DIR}")
print(f"Qdrant:     {QDRANT_URL}")
print(f"Collection: {COLLECTION_NAME}")


Embeddings: C:\Users\user\Documents\GitHub\clara-rag-pt\data\embeddings
Qdrant:     http://localhost:6333
Collection: clara_bge_m3


## 2. Project Paths and Qdrant Configuration

The notebook defines the location of the generated embedding files and the configuration required to communicate with Qdrant.

The vector collection name and Qdrant endpoint must remain consistent with the runtime configuration used by the CLARA retrieval application.

In [3]:
embedding_files = sorted(
    EMBEDDINGS_DIR.glob("*_embeddings.json")
)

print(
    f"Foram encontrados {len(embedding_files)} ficheiros de embeddings:\n"
)

for path in embedding_files:
    print(f"- {path.name}")

if not embedding_files:
    raise FileNotFoundError(
        "Não foram encontrados ficheiros de embeddings. "
        "Executa primeiro o Notebook 4."
    )


Foram encontrados 10 ficheiros de embeddings:

- CIMI_embeddings.json
- CIRC_embeddings.json
- CIRS_embeddings.json
- CIS_embeddings.json
- CIVA_embeddings.json
- EBF_embeddings.json
- LGT_embeddings.json
- OE2026_embeddings.json
- RGIT_embeddings.json
- RITI_embeddings.json


## Verificar ligação ao Qdrant


## 4. Qdrant Availability Check

Before creating or modifying the vector collection, the notebook verifies that the Qdrant service is reachable.

This prevents the indexing workflow from proceeding when the vector database is unavailable or incorrectly configured.

In [4]:
def check_qdrant() -> bool:
    try:
        response = requests.get(
            f"{QDRANT_URL}/collections",
            timeout=10
        )

        response.raise_for_status()
        return True

    except Exception as e:
        print(f"Erro ao contactar o Qdrant: {e}")
        return False


qdrant_available = check_qdrant()

print(
    "Qdrant disponível."
    if qdrant_available
    else "Qdrant indisponível."
)


Qdrant disponível.


## Determinar dimensão dos embeddings

A dimensão da coleção deve corresponder exatamente à dimensão dos vetores produzidos pelo modelo.


## 5. Determining the Vector Dimension

Qdrant collections require a fixed vector dimensionality.

The dimension is obtained from the generated embeddings rather than being assumed manually. This ensures that the collection configuration matches the vectors produced by the selected embedding model.

In [5]:
with open(
    embedding_files[0],
    "r",
    encoding="utf-8"
) as f:
    first_document = json.load(f)

VECTOR_SIZE = first_document["embedding_dimension"]

print(
    f"Dimensão dos vetores: {VECTOR_SIZE}"
)

print(
    f"Modelo: {first_document['embedding_model']}"
)


Dimensão dos vetores: 1024
Modelo: bge-m3


## Verificar se a coleção já existe


## 4. Qdrant Availability Check

Before creating or modifying the vector collection, the notebook verifies that the Qdrant service is reachable.

This prevents the indexing workflow from proceeding when the vector database is unavailable or incorrectly configured.

In [6]:
def collection_exists(
    collection_name: str
) -> bool:

    response = requests.get(
        f"{QDRANT_URL}/collections/{collection_name}",
        timeout=10
    )

    if response.status_code == 200:
        return True

    if response.status_code == 404:
        return False

    response.raise_for_status()

    return False


exists = collection_exists(
    COLLECTION_NAME
)

print(
    f"A coleção '{COLLECTION_NAME}' "
    f"{'já existe' if exists else 'não existe'}."
)


A coleção 'clara_bge_m3' não existe.


## Criar a coleção

Se quiseres recriar a coleção de raiz, define `RECREATE_COLLECTION = True`.


## 6. Preparing the Qdrant Collection

This block prepares the vector collection used by CLARA.

When appropriate, an existing collection can be removed or recreated so that its vector configuration matches the current embedding dataset.

Recreating a collection deletes its existing indexed points, so this operation should be performed deliberately when rebuilding the corpus.

In [7]:
RECREATE_COLLECTION = True

if RECREATE_COLLECTION and collection_exists(
    COLLECTION_NAME
):
    response = requests.delete(
        f"{QDRANT_URL}/collections/{COLLECTION_NAME}",
        timeout=30
    )

    response.raise_for_status()

    print(
        f"Coleção '{COLLECTION_NAME}' eliminada."
    )


### 6.1 Creating the Vector Collection

The CLARA Qdrant collection is created with the vector dimensionality and similarity metric required by the embedding model.

All indexed legal chunks must use the same vector configuration so that semantic similarity scores are comparable during retrieval.

In [8]:
if not collection_exists(
    COLLECTION_NAME
):

    payload = {
        "vectors": {
            "size": VECTOR_SIZE,
            "distance": "Cosine"
        }
    }

    response = requests.put(
        f"{QDRANT_URL}/collections/{COLLECTION_NAME}",
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    print(
        f"Coleção '{COLLECTION_NAME}' criada."
    )

else:

    print(
        f"Coleção '{COLLECTION_NAME}' já existe."
    )


Coleção 'clara_bge_m3' criada.


## Funções auxiliares para indexação


## 7. Building Qdrant Points

Each embedding record is transformed into a Qdrant point.

A point combines three elements:

- a unique point identifier;
- the semantic vector;
- a payload containing the legal metadata and source text.

The payload is essential because retrieval must return not only similar vectors but also the legal information required to construct the RAG context and identify the original source.

In [9]:
def stable_point_id(chunk_id: str) -> str:
    """
    Converte o chunk_id num UUID determinístico.

    O mesmo chunk_id gera sempre o mesmo UUID.
    """

    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            chunk_id
        )
    )


## 7. Building Qdrant Points

Each embedding record is transformed into a Qdrant point.

A point combines three elements:

- a unique point identifier;
- the semantic vector;
- a payload containing the legal metadata and source text.

The payload is essential because retrieval must return not only similar vectors but also the legal information required to construct the RAG context and identify the original source.

In [10]:
def build_qdrant_point(
    chunk: dict
) -> dict:
    """Converte um chunk num ponto Qdrant."""

    hierarchy = chunk.get(
        "hierarchy",
        {}
    )

    payload = {
        "chunk_id": chunk["chunk_id"],
        "document_id": chunk["document_id"],
        "article_index": chunk.get("article_index"),
        "article": chunk["article"],
        "article_number": chunk["article_number"],
        "article_title": chunk.get("article_title"),
        "page_start": chunk["page_start"],
        "page_end": chunk["page_end"],
        "chunk_index": chunk["chunk_index"],
        "total_chunks": chunk["total_chunks"],
        "word_count": chunk["word_count"],
        "text": chunk["text"],
        "embedding_text": chunk["embedding_text"],
        "part": hierarchy.get("part"),
        "book": hierarchy.get("book"),
        "title": hierarchy.get("title"),
        "chapter": hierarchy.get("chapter"),
        "section": hierarchy.get("section"),
        "subsection": hierarchy.get("subsection")
    }

    return {
        "id": stable_point_id(
            chunk["chunk_id"]
        ),
        "vector": chunk["embedding"],
        "payload": payload
    }


## Testar a conversão de um chunk


## 5. Determining the Vector Dimension

Qdrant collections require a fixed vector dimensionality.

The dimension is obtained from the generated embeddings rather than being assumed manually. This ensures that the collection configuration matches the vectors produced by the selected embedding model.

In [11]:
test_chunk = first_document["chunks"][0]

test_point = build_qdrant_point(
    test_chunk
)

print("Point ID:")
print(test_point["id"])

print("\nPayload:")
print(
    json.dumps(
        test_point["payload"],
        ensure_ascii=False,
        indent=2
    )[:2500]
)

print(
    f"\nDimensão do vetor: "
    f"{len(test_point['vector'])}"
)


Point ID:
560c8bcf-196d-5390-a9d8-1cd3b65cf7cd

Payload:
{
  "chunk_id": "CIMI_ART_0001_CHUNK_001",
  "document_id": "CIMI",
  "article_index": 1,
  "article": "Artigo 13.º",
  "article_number": "13.º",
  "article_title": "Elementos para avaliações",
  "page_start": 8,
  "page_end": 8,
  "chunk_index": 1,
  "total_chunks": 1,
  "word_count": 113,
  "text": "1 - Os elementos referidos nas alíneas a) a d) do n.º 1 do artigo 62.º do CIMI deverão ser aprovados no prazo de 180 dias a contar da data da publicação do presente diploma. 2 - As plantas previstas na alínea b) do artigo 128.º do CIMI devem ser remetidas aos serviços de finanças da área do município no prazo de 30 dias a contar da data da publicação do presente diploma. 3 - As câmaras municipais devem colaborar com os serviços competentes da Direcção-Geral dos Impostos na elaboração das propostas de fixação dos elementos de avaliação, as quais devem ser-lhes previamente remetidas para que se pronunciem no prazo de 15 dias.",
  "emb

## Inserção em batches


### 6.1 Creating the Vector Collection

The CLARA Qdrant collection is created with the vector dimensionality and similarity metric required by the embedding model.

All indexed legal chunks must use the same vector configuration so that semantic similarity scores are comparable during retrieval.

In [12]:
def upsert_points(
    points: list[dict],
    batch_size: int = 64
):
    """Insere pontos no Qdrant em batches."""

    total = len(points)

    for start in range(
        0,
        total,
        batch_size
    ):

        batch = points[
            start:start + batch_size
        ]

        response = requests.put(
            (
                f"{QDRANT_URL}/collections/"
                f"{COLLECTION_NAME}/points"
            ),
            params={
                "wait": "true"
            },
            json={
                "points": batch
            },
            timeout=120
        )

        response.raise_for_status()

        print(
            f"  {min(start + batch_size, total)}/{total}"
        )


## Indexar todos os documentos


## 2. Project Paths and Qdrant Configuration

The notebook defines the location of the generated embedding files and the configuration required to communicate with Qdrant.

The vector collection name and Qdrant endpoint must remain consistent with the runtime configuration used by the CLARA retrieval application.

In [13]:
results = []

for embedding_path in embedding_files:

    print(
        f"\nA indexar {embedding_path.name}..."
    )

    try:

        with open(
            embedding_path,
            "r",
            encoding="utf-8"
        ) as f:
            document = json.load(f)

        document_id = document[
            "document_id"
        ]

        points = [
            build_qdrant_point(chunk)
            for chunk in document["chunks"]
        ]

        upsert_points(
            points,
            batch_size=BATCH_SIZE
        )

        results.append({
            "document_id": document_id,
            "points": len(points),
            "status": "OK"
        })

        print(
            f"{document_id}: OK "
            f"({len(points)} pontos)"
        )

    except Exception as e:

        results.append({
            "document_id": embedding_path.stem,
            "points": None,
            "status": f"ERROR: {e}"
        })

        print(
            f"ERRO: {e}"
        )



A indexar CIMI_embeddings.json...
  64/226
  128/226
  192/226
  226/226
CIMI: OK (226 pontos)

A indexar CIRC_embeddings.json...
  64/305
  128/305
  192/305
  256/305
  305/305
CIRC: OK (305 pontos)

A indexar CIRS_embeddings.json...
  64/370
  128/370
  192/370
  256/370
  320/370
  370/370
CIRS: OK (370 pontos)

A indexar CIS_embeddings.json...
  64/130
  128/130
  130/130
CIS: OK (130 pontos)

A indexar CIVA_embeddings.json...
  64/354
  128/354
  192/354
  256/354
  320/354
  354/354
CIVA: OK (354 pontos)

A indexar EBF_embeddings.json...
  64/279
  128/279
  192/279
  256/279
  279/279
EBF: OK (279 pontos)

A indexar LGT_embeddings.json...
  64/155
  128/155
  155/155
LGT: OK (155 pontos)

A indexar OE2026_embeddings.json...
  64/404
  128/404
  192/404
  256/404
  320/404
  384/404
  404/404
OE2026: OK (404 pontos)

A indexar RGIT_embeddings.json...
  64/165
  128/165
  165/165
RGIT: OK (165 pontos)

A indexar RITI_embeddings.json...
  64/168
  128/168
  168/168
RITI: OK (168 

## 10. Indexing Results

The indexing results are represented in tabular form to provide a document-level overview of the operation.

This makes it possible to confirm which legal documents were indexed successfully and how many vector points were processed for each source.

In [14]:
df_results = pd.DataFrame(
    results
)

df_results


,document_id,points,status
0,CIMI,226,OK
1,CIRC,305,OK
2,CIRS,370,OK
3,CIS,130,OK
4,CIVA,354,OK
5,EBF,279,OK
6,LGT,155,OK
7,OE2026,404,OK
8,RGIT,165,OK
9,RITI,168,OK


## Contar os pontos da coleção


## 11. Collection Point Count Validation

After indexing, the number of points stored in Qdrant is checked.

The indexed point count should correspond to the number of generated legal chunks and embeddings. Differences may indicate incomplete uploads, identifier collisions, or preprocessing inconsistencies.

In [15]:
response = requests.post(
    (
        f"{QDRANT_URL}/collections/"
        f"{COLLECTION_NAME}/points/count"
    ),
    json={
        "exact": True
    },
    timeout=30
)

response.raise_for_status()

count_data = response.json()

indexed_points = count_data[
    "result"
]["count"]

print(
    f"Pontos indexados: {indexed_points}"
)


Pontos indexados: 2556


### Indexing Step 16

This block performs an additional operation in the Qdrant indexing workflow.

It contributes to converting the generated embedding dataset into a persistent and searchable vector collection while preserving the legal metadata required by the CLARA retrieval pipeline.

In [16]:
expected_points = int(
    df_results["points"]
    .fillna(0)
    .sum()
)

print(
    f"Pontos esperados: {expected_points}"
)

print(
    f"Pontos no Qdrant: {indexed_points}"
)

assert (
    indexed_points
    ==
    expected_points
), (
    "O número de pontos no Qdrant "
    "não corresponde ao esperado."
)

print(
    "Contagem validada."
)


Pontos esperados: 2556
Pontos no Qdrant: 2556
Contagem validada.


## Criar índices de payload

Estes índices tornam mais eficientes filtros posteriores por documento e artigo.


### 6.1 Creating the Vector Collection

The CLARA Qdrant collection is created with the vector dimensionality and similarity metric required by the embedding model.

All indexed legal chunks must use the same vector configuration so that semantic similarity scores are comparable during retrieval.

In [17]:
payload_indexes = [
    ("document_id", "keyword"),
    ("article", "keyword"),
    ("article_number", "keyword")
]

for field_name, field_schema in payload_indexes:

    response = requests.put(
        (
            f"{QDRANT_URL}/collections/"
            f"{COLLECTION_NAME}/index"
        ),
        json={
            "field_name": field_name,
            "field_schema": field_schema
        },
        timeout=30
    )

    # O Qdrant pode devolver erro se o índice já existir.
    if response.status_code in (200, 201):
        print(
            f"Índice criado: {field_name}"
        )
    else:
        print(
            f"{field_name}: "
            f"{response.status_code} "
            f"{response.text[:200]}"
        )


Índice criado: document_id
Índice criado: article
Índice criado: article_number


## Ler alguns pontos para validação


## 12. Inspecting Indexed Points

A sample of stored Qdrant points is retrieved using the collection scroll operation.

Inspecting indexed records verifies that vectors and legal payload metadata were persisted correctly and are available for subsequent retrieval.

In [18]:
response = requests.post(
    (
        f"{QDRANT_URL}/collections/"
        f"{COLLECTION_NAME}/points/scroll"
    ),
    json={
        "limit": 5,
        "with_payload": True,
        "with_vector": False
    },
    timeout=30
)

response.raise_for_status()

sample_points = response.json()[
    "result"
]["points"]

for point in sample_points:

    payload = point["payload"]

    print("=" * 80)
    print("ID:", point["id"])
    print("Documento:", payload["document_id"])
    print("Artigo:", payload["article"])
    print("Título:", payload["article_title"])
    print(
        "Páginas:",
        payload["page_start"],
        "-",
        payload["page_end"]
    )
    print()
    print(
        payload["text"][:500]
    )


ID: 00154382-3378-5633-9034-f66dfe57dfef
Documento: CIVA
Artigo: Artigo 13.º
Título: None
Páginas: 23 - 24

1 - ... 2 - ... 3 - A isenção referida na alínea d) do n.º 1 não é aplicável a: a) ... b) ... - 23 –
ID: 001bba2c-22fb-5af0-ba83-828c461ba39f
Documento: CIMI
Artigo: Artigo 13.º
Título: Inscrição nas matrizes
Páginas: 32 - 33

do respectivo titular; d) À eliminação na matriz dos prédios demolidos, após informação dos serviços relativa ao termo da demolição; e) À inscrição do valor patrimonial tributário definitivo determinado nos termos do presente Código. f) À atualização da matriz nos casos da comunicação por parte do município prevista na alínea b) do n.º 3 do artigo 6.º (Redação da Lei n.º 56/2023, de 06/10) 4 - As inscrições ou actualizações matriciais devem referir o ano em que tenham sido efectuadas, bem como o
ID: 0023cbe9-dce8-563a-b188-1b47cf366aef
Documento: OE2026
Artigo: Artigo 8.º
Título: Alterações orçamentais
Páginas: 6 - 11

alterações orçamentais decorrentes da 

## Estatísticas finais


## 5. Determining the Vector Dimension

Qdrant collections require a fixed vector dimensionality.

The dimension is obtained from the generated embeddings rather than being assumed manually. This ensures that the collection configuration matches the vectors produced by the selected embedding model.

In [19]:
print(
    f"Collection: {COLLECTION_NAME}"
)

print(
    f"Vector size: {VECTOR_SIZE}"
)

print(
    "Distance: Cosine"
)

print(
    f"Documentos indexados: "
    f"{len(df_results[df_results['status'] == 'OK'])}"
)

print(
    f"Pontos indexados: {indexed_points}"
)


Collection: clara_bge_m3
Vector size: 1024
Distance: Cosine
Documentos indexados: 10
Pontos indexados: 2556


## Next Step

The legal corpus is now indexed in Qdrant and can be queried through semantic similarity search.

The next notebook is:

```text
06_vector_search.ipynb
```

It generates an embedding for a natural-language question using **BGE-M3**, searches the `clara_bge_m3` collection, and inspects the most relevant legal chunks returned by Qdrant.

This marks the transition from the **ingestion pipeline** to the **retrieval pipeline** of CLARA.
